<a href="https://colab.research.google.com/github/Soha-Waseem/FlyRank_ML_Week1/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Soha-Waseem/FlyRank_ML_Week1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os
import getpass
import duckdb

HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

HF_TOKEN = HF_TOKEN or getpass.getpass("Paste HF token: ")

In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


## 1. Unit of analysis + time window

**One row**

One row represents one search query for one landing page on one day.

**Time Window**

This notebook uses data from March 2026 (month = '2026-03'). A mid-panel month is used to avoid using the final month (June 2026), which is reserved as a future evaluation period.

**Prediction Goal**

The objective is to predict or rank future organic search performance using historical Search Console data.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

## Field Classification

### Features
- gsc_clicks
- gsc_impressions
- gsc_avg_position
- ga4_sessions
- ga4_engaged_sessions

### Label
Future content performance (declining or not).

### Context
- client_hash_id
- content_hash_id
- report_date

### Excluded
- June 2026 data (reserved for testing)
- Future-derived columns (to avoid leakage)
- Hash IDs as model features (used only for grouping and joins)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
LIMIT 5
""").df()


,report_date,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,0,20,3.350000,<NA>,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,0,1,0.000000,<NA>,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,1,125,4.928000,<NA>,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,0,7,4.000000,<NA>,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,0,11,2.272727,<NA>,<NA>


## 3. Verify it with queries (grain, counts, missing values, windows)

### Query 1 – Verify the grain

This query checks that there is only one row for each combination of report date, client, and content item.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


### Query 2 – Verify row count and date range

This confirms the size of the selected month and its reporting period.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS rows,
    MIN(report_date) AS first_day,
    MAX(report_date) AS last_day
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
""").df()

,rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


### Query 3 – Availability

Only rows where GA4 data is available are kept.

In [ ]:
con.sql(f"""
SELECT
    COUNT(*) AS usable_rows
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND ga4_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,usable_rows
0,413966


## 4. Data limits

- Different clients have different amounts of historical data.
- GA4 metrics are unavailable before each client's `ga4_data_start`.
- June 2026 is excluded because it is the final month and should remain an unseen test period.
- The warehouse measures search and analytics performance but cannot explain the reasons behind ranking changes.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

features = con.sql(f"""
SELECT
    gsc_clicks,
    gsc_impressions,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
AND ga4_data_available IS TRUE
LIMIT 1000
""").df()

features.head()

,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,ga4_engaged_sessions
0,0,0,NaN,1,0
1,0,0,NaN,1,0
2,0,0,NaN,1,0
3,0,0,NaN,1,0
4,0,0,NaN,1,0


### Feature Availability

- **gsc_clicks** – Knowable because daily Search Console clicks are available after the reporting day.
- **gsc_impressions** – Knowable because impressions are recorded daily in Search Console.
- **gsc_avg_position** – Knowable because ranking data is finalized after the reporting day.
- **ga4_sessions** – Knowable when `ga4_data_available` is TRUE.
- **ga4_engaged_sessions** – Knowable alongside other GA4 metrics after processing.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.